In [1]:
# --- KOMÓRKA 1: IMPORTY I KONFIGURACJA ---
import cv2
import numpy as np
import tensorflow as tf
import time

# Wyłączamy ostrzeżenia OneDNN (opcjonalne, dla czystości logów)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# USTAWIENIA
IMG_SIZE = 400             # Rozmiar wejściowy Twoich sieci
PADDING = 20               # Margines wokół dłoni (zwiększ jeśli ucina palce)
CONFIDENCE_THRESHOLD = 0.7 # Minimalna pewność klasyfikacji, by zmienić kolor na zielony
CAM_SOURCE = 0             # 0 to zazwyczaj domyślna kamera w laptopie

# LISTA KLAS (Musi być w tej samej kolejności co podczas trenowania!)
# CLASSES = ['Yes', 'No', 'Hello', 'I love You', 'Thank You']

print("Biblioteki załadowane.")

2026-01-30 16:25:29.114266: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 16:25:29.792991: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 16:25:44.840411: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Biblioteki załadowane.


In [2]:
from pathlib import Path

DATASET_PATH = "./Sign Language Detection/babeczka"
root = Path(DATASET_PATH)
assert root.exists(), f"Dataset path not found: {DATASET_PATH}. Change DATASET_PATH to correct folder."
print(root)

Sign Language Detection/babeczka


In [3]:
from pathlib import Path
import yaml

data_path = root / "data.yaml"

with data_path.open("r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

CLASSES = data["names"]          # list of class names
amount = data.get("nc", len(CLASSES))
print("names:", CLASSES)

names: ['hello', 'iloveyou', 'no', 'thanks', 'yes']


In [4]:
# --- KOMÓRKA 2: ŁADOWANIE MODELI ---

# Ścieżki do modeli - upewnij się, że są poprawne
DETECTOR_PATH = 'hand_detector_best.keras'
CLASSIFIER_PATH = 'gesture_classifier.keras' # Twój model ResNet lub MobileNet

try:
    print("Ładowanie detektora...")
    detector = tf.keras.models.load_model(DETECTOR_PATH)
    
    print("Ładowanie klasyfikatora...")
    classifier = tf.keras.models.load_model(CLASSIFIER_PATH)
    
    print("Modele załadowane pomyślnie!")
except Exception as e:
    print(f"BŁĄD PODCZAS ŁADOWANIA MODELI: {e}")
    print("Sprawdź ścieżki do plików .keras!")

Ładowanie detektora...


I0000 00:00:1769786776.164960   45671 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9515 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


Ładowanie klasyfikatora...
Modele załadowane pomyślnie!


In [7]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def crop_hand(image, model):
    # 1. Przygotuj obraz
    original_img = image.copy()
    img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
    input_tensor = np.expand_dims(img_resized, axis=0)

    # 2. Predykcja ramki
    box = model.predict(input_tensor)[0] # [x, y, w, h] (znormalizowane)
    
    # 3. Przelicz na piksele
    h_orig, w_orig, _ = original_img.shape
    x = int(box[0] * w_orig)
    y = int(box[1] * h_orig)
    w = int(box[2] * w_orig)
    h = int(box[3] * h_orig)

    # Zabezpieczenia (żeby nie wyjść poza obraz przy błędzie modelu)
    x = max(0, x)
    y = max(0, y)
    
    # 4. Wycięcie (Crop)
    # Dodajemy mały margines (padding), żeby nie uciąć palców
    padding = 20
    crop = original_img[y-padding : y+h+padding, x-padding : x+w+padding]
    
    return crop, (x, y, w, h)


def predict_full_pipeline(image, detector_model, classifier_model, classes_list):
    img_bgr = image.copy()
    hand_crop, coords = crop_hand(img_bgr.copy(), detector_model)
    x_pad = coords[0]
    y_pad = coords[1]
    w_pad = coords[2]
    h_pad = coords[3]

    # Zabezpieczenie przed pustym wycinkiem (gdyby detektor zawiódł)
    if hand_crop.size == 0:
        print("Nie udało się wyciąć dłoni (błąd detektora).")
        return

    # =========================================================
    # KROK 2: KLASYFIKACJA (Jaki to gest?)
    # =========================================================

    # Preprocessing dla klasyfikatora
    img_cls = cv2.resize(hand_crop, (IMG_SIZE, IMG_SIZE)) / 255.0
    img_cls_batch = np.expand_dims(img_cls, axis=0)

    # Predykcja gestu
    predictions = classifier_model.predict(img_cls_batch, verbose=0)[0]
    class_idx = np.argmax(predictions)       # Indeks najwyższego wyniku
    confidence = predictions[class_idx]      # Pewność (0.0 - 1.0)
    
    predicted_label = classes_list[class_idx]
    
    print(f"Predykcja: {predicted_label} z pewnością {confidence*100:.2f}%")

    # =========================================================
    # KROK 3: WIZUALIZACJA
    # =========================================================
    
    # Kolor ramki zależny od pewności (Zielony = pewny, Żółty = niepewny)
    color = (0, 255, 0) if confidence > 0.7 else (0, 255, 255)

    # Rysujemy ramkę na ORYGINALNYM obrazie BGR
    cv2.rectangle(img_bgr, (x_pad, y_pad), (x_pad + w_pad, y_pad + h_pad), color, 2)

    # Przygotowanie tekstu: "I love You (98%)"
    text = f"{predicted_label} ({confidence*100:.1f}%)"
    
    # Tło pod napisem (dla czytelności)
    (text_w, text_h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(img_bgr, (x_pad, y_pad - 25), (x_pad + text_w, y_pad), color, -1)
    
    # Napis
    cv2.putText(img_bgr, text, (x_pad, y_pad - 5), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

    # Pokazanie wyniku
    cropped_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(cropped_img_rgb)
    # cv2.imshow("Wynik Pipeline", img_bgr)
    
    # Opcjonalnie: Pokaż też, co "widział" klasyfikator (wycinek)
    cv2.imshow("Co widzi klasyfikator", cv2.cvtColor(hand_crop, cv2.COLOR_RGB2BGR))
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    
    
    # Kolor ramki: Zielony jeśli pewny, Żółty jeśli niepewny
    color = (0, 255, 0) if confidence > CONFIDENCE_THRESHOLD else (0, 255, 255)
    
    # Rysujemy na obrazie BGR (debug_frame)
    cv2.rectangle(img_bgr, (x_pad, y_pad), (x_pad, y_pad), color, 2)
    
    # Tło pod napis
    label_text = f"{label} ({confidence*100:.1f}%)"
    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(img_bgr, (x_pad, y_pad - 30), (x_pad + text_w, y_pad), color, -1)
    
    # Napis
    cv2.putText(img_bgr, label_text, (x_pad, y_pad - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    
    img_bgr[0:100, 0:100] = cv2.resize(cv2.cvtColor(hand_crop, cv2.COLOR_RGB2BGR), (100,100))
    # Wyświetlenie klatki
    cv2.imshow('Hand Sign Recognition', img_bgr)

In [11]:
def run_live_inference():
    cap = cv2.VideoCapture(CAM_SOURCE)
    
    # Ustawienie rozdzielczości kamery (opcjonalne, dla wydajności)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    if not cap.isOpened():
        print("Nie można otworzyć kamery!")
        return

    print("Kamera uruchomiona. Naciśnij 'q', aby wyjść.")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Kopia do wyświetlania (zeby rysowac na czystym)
        debug_frame = frame.copy()
        h_orig, w_orig, _ = frame.shape

        predict_full_pipeline(frame, detector, classifier, CLASSES)
        
        # Wyświetlenie klatki
        cv2.imshow('Hand Sign Recognition', debug_frame)

        # Wyjście klawiszem 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [27]:
run_live_inference()

Nie można otworzyć kamery!


[ WARN:0@758.952] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@758.952] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 0
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@758.952] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
